# 07_hnsw: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/07_hnsw/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Walk a layered product graph and inspect its edges.

Concepts: Walk a layered product graph and inspect its edges.
Run: uv run python modules/07_hnsw/examples/01_from_scratch.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

from common.index_lessons import unit_catalog
from tinyvec.hnsw import HNSW


def main():
    products, matrix, query = unit_catalog(220)
    graph = HNSW(matrix, M=6, efConstruction=24, efSearch=24)
    print("layer populations", [len(layer) for layer in graph.layers], "entry", graph.entry)
    candidates = graph.candidates(query, 10)
    print(
        "entry product",
        products.iloc[graph.entry].title,
        "found",
        products.iloc[candidates[:3]].title.tolist(),
    )
    assert all(len(edges) <= 6 for layer in graph.layers for edges in layer.values())


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/07_hnsw/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Compare FAISS and USearch graph candidates.

    Concepts: Compare FAISS and USearch graph candidates.
    Run: uv run --extra libs python modules/07_hnsw/examples/02_production.py
    Expected: Inspect the printed IDs or measurements and the checked invariant.
    CPU: seconds on the included small workload. Uses local data; no Docker or model download.
    """

    # course-extra: libs
    import faiss
    import numpy as np
    from usearch.index import Index

    from common.index_lessons import unit_catalog

    def main():
        products, matrix, q = unit_catalog(640)
        faiss.omp_set_num_threads(1)
        index = faiss.IndexHNSWFlat(matrix.shape[1], 8, faiss.METRIC_INNER_PRODUCT)
        index.hnsw.efConstruction = 32
        index.hnsw.efSearch = 24
        index.add(matrix)
        _, rows = index.search(q[None, :], 10)
        native = Index(
            ndim=matrix.shape[1],
            metric="cos",
            dtype="f32",
            connectivity=8,
            expansion_add=32,
            expansion_search=24,
        )
        native.add(np.arange(len(matrix), dtype=np.uint64), matrix, threads=1)
        matches = native.search(q, 10, threads=1)
        print(
            "FAISS",
            products.iloc[rows[0]].id.tolist(),
            "USearch",
            products.iloc[matches.keys].id.tolist(),
        )
        assert len(matches.keys) == len(rows[0]) == 10

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/07_hnsw/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure beam width and graph payload.

Concepts: Measure beam width and graph payload.
Run: uv run python modules/07_hnsw/examples/03_measure.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

from common.index_lessons import trial
from common.teaching import evidence


def main():
    observations = [
        trial("hnsw", M=6, efConstruction=32, efSearch=e, oversample=1) for e in (10, 24, 64)
    ]
    print(observations)
    evidence("lesson-07_hnsw", {"observations": observations}, tier=2)


if __name__ == "__main__":
    main()

## 04_hnswlib

Source: `modules/07_hnsw/examples/04_hnswlib.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("hnswlib") is None:
    print("SKIP native comparison: --extra hnswlib")
else:
    """Build and search the optional native hnswlib graph.

    Concepts: Build and search the optional native hnswlib graph.
    Run: uv run --extra hnswlib python modules/07_hnsw/examples/04_hnswlib.py
    Expected: Inspect the printed IDs or measurements and the checked invariant.
    CPU: seconds on the included small workload. Uses local data; no Docker or model download.
    """

    # course-extra: hnswlib
    from common.index_lessons import trial

    def main():
        result = trial("hnswlib", M=8, efConstruction=40, efSearch=32)
        print(result)
        assert len(result["ids"]) == 10
        import hnswlib

        from common.index_lessons import unit_catalog

        _, matrix, _ = unit_catalog(160)
        graph = hnswlib.Index(space="cosine", dim=matrix.shape[1])
        graph.init_index(max_elements=len(matrix), M=8, ef_construction=32, random_seed=42)
        graph.add_items(matrix, num_threads=1)
        graph.mark_deleted(0)
        labels, _ = graph.knn_query(matrix[0], k=10, num_threads=1)
        assert 0 not in labels[0]
        print("tombstoned row 0; surviving neighbors", labels[0])

    if __name__ == "__main__":
        main()

## 05_deletions

Source: `modules/07_hnsw/examples/05_deletions.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Retire an item from a native USearch graph.

    Run: uv run --extra libs python modules/07_hnsw/examples/05_deletions.py
    Expected: compare the reported ranking and verify the final assertion.
    CPU: seconds, included data, no Docker or model downloads.
    """

    # course-extra: libs
    import numpy as np
    from usearch.index import Index

    from common.index_lessons import unit_catalog

    def main():
        _, matrix, _ = unit_catalog(100)
        index = Index(ndim=matrix.shape[1], metric="cos", dtype="f32")
        index.add(np.arange(len(matrix), dtype=np.uint64), matrix, threads=1)
        index.remove(0)
        after = index.search(matrix[0], 10, threads=1).keys.tolist()
        print("survivors", after)
        assert 0 not in after and len(index) == 99

    if __name__ == "__main__":
        main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.index_lessons import trial

print(trial("hnsw", M=4, efConstruction=20, efSearch=10))

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.